In [26]:
import os
import numpy as np
import librosa
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Reshape
from sklearn.metrics import accuracy_score, classification_report


In [27]:
def extract_features(file_path, duration=10.0):
    audio, sr = librosa.load(file_path, duration=duration)
    spectrogram = np.abs(librosa.stft(audio))
    return spectrogram

In [28]:
non_music_path = "dataset1/non_music"
music_path = "dataset1/music"

In [29]:
non_music_files = [os.path.join(non_music_path, f) for f in os.listdir(non_music_path)]
music_files = [os.path.join(music_path, f) for f in os.listdir(music_path)]

In [30]:
def extract_features(file_path, chunk_length=5.0, max_length=220500):
    audio, sr = librosa.load(file_path)  # Load entire audio file
    total_frames = int(chunk_length * sr)  # Calculate frames per chunk
    num_chunks = int(np.ceil(len(audio) / total_frames))  # Calculate number of chunks
    
    spectrograms = []
    for i in range(num_chunks):
        start_frame = i * total_frames
        end_frame = min(start_frame + total_frames, len(audio))
        chunk_audio = audio[start_frame:end_frame]
        
        spectrogram = np.abs(librosa.stft(chunk_audio))
        if spectrogram.shape[1] < max_length:
            # If shorter than max_length, pad with zeros
            pad_width = max_length - spectrogram.shape[1]
            spectrogram = np.pad(spectrogram, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            # If longer than max_length, truncate
            spectrogram = spectrogram[:, :max_length]
        
        spectrograms.append(spectrogram)
    
    return np.array(spectrograms)


In [31]:
non_music_files = [os.path.join(non_music_path, f) for f in os.listdir(non_music_path)]

In [32]:
music_files = [os.path.join(music_path, f) for f in os.listdir(music_path)]

In [33]:
batch_size = 10
max_length = 220500

In [34]:
def process_files(file_paths):
    features = []
    for i in range(0, len(file_paths), batch_size):
        batch_files = file_paths[i:i+batch_size]
        batch_features = [extract_features(f) for f in batch_files]
        features.extend(batch_features)
    return np.array(features)

In [35]:
non_music_features = process_files(non_music_files)


MemoryError: Unable to allocate 862. MiB for an array with shape (1, 1025, 220500) and data type float32

In [ ]:
music_features = process_files(music_files)

In [ ]:
non_music_labels = np.zeros(len(non_music_files))
music_labels = np.ones(len(music_files))

In [ ]:
X = np.concatenate((non_music_features, music_features))
y = np.concatenate((non_music_labels, music_labels))


In [ ]:
input_shape = X.shape[1:]  # Shape of one spectrogram
X = X.reshape(X.shape[0], *input_shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
model = Sequential([
    Reshape(input_shape, input_shape=input_shape),
    Flatten(),
    Dense(2048, activation='relu'),
    Dense(1024, activation='relu'),
    Dense(512, activation='relu'),
    Dense(256, activation='relu'),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

In [ ]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])


In [ ]:
model.fit(X_train, y_train, epochs=10, batch_size=1024, validation_split=0.2)


In [ ]:
y_pred = (model.predict(X_test) > 0.5).astype("int32")
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

In [ ]:
print("Classification Report:")
print(classification_report(y_test, y_pred))